# Train the GILGAMESH X-ray threat detector

This notebook trains a YOLO object detector on real baggage X-ray images and exports the two files the console loads in the browser: `model.onnx` and `model.json`.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

1. Install the tools.
2. Get a dataset (Roboflow, a zip you upload, or a folder you already have).
3. Pick which classes count as threats.
4. Train, then check accuracy.
5. Export and download `xray-model.zip`. Unzip it into `public/models/xray/` in the repo.

Expect roughly 30–60 minutes for 50 epochs on a few thousand images with the nano model.

> **Licenses:** most public X-ray datasets (SIXray, PIDray, OPIXray, HiXray) are for research use only. That's fine for a hackathon demo but not for a commercial product. Check the license of whatever you train on.

In [ ]:
!pip install -q ultralytics roboflow onnx onnxslim onnxruntime
import os, sys
IN_COLAB = "google.colab" in sys.modules
WORK = "/content" if IN_COLAB else os.path.abspath("xray_work")
os.makedirs(WORK, exist_ok=True)
import ultralytics
ultralytics.checks()

## 2. Get a dataset

Pick **one** source in the next cell.

**Option A: Roboflow (easiest).** Go to [universe.roboflow.com](https://universe.roboflow.com), search for `x-ray baggage` or `prohibited items x-ray`, and open an object-detection dataset with a few thousand images. Click **Download Dataset**, choose format **YOLOv11** (YOLOv8 also works), then **Show download code**. Copy your API key, workspace, project and version from that snippet into the cell below. A free account is enough.

**Option B: zip upload.** A zip of a YOLO-format dataset: a `data.yaml` plus `train/` and `valid/` folders, each with `images/` and `labels/`.

**Option C: a folder you already have**, for example on Google Drive. Set `DATA_YAML` to its `data.yaml`.

In [ ]:
DATASET_SOURCE = "roboflow"   # "roboflow", "zip", or "path"
DATA_YAML = None              # for "path": e.g. "/content/drive/MyDrive/xray/data.yaml"

# --- Option A: paste the values from Roboflow's "Show download code" ---
ROBOFLOW_API_KEY = "PASTE_YOUR_API_KEY"
ROBOFLOW_WORKSPACE = "workspace-id"
ROBOFLOW_PROJECT = "project-id"
ROBOFLOW_VERSION = 1

if DATASET_SOURCE == "roboflow":
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    dataset = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT).version(ROBOFLOW_VERSION).download(
        "yolov11", location=os.path.join(WORK, "dataset")
    )
    DATA_YAML = os.path.join(dataset.location, "data.yaml")

elif DATASET_SOURCE == "zip":
    import glob, zipfile
    from google.colab import files
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    zipfile.ZipFile(zip_name).extractall(os.path.join(WORK, "dataset"))
    found = sorted(glob.glob(os.path.join(WORK, "dataset", "**", "data.yaml"), recursive=True), key=len)
    assert found, "No data.yaml in that zip. It needs data.yaml plus train/ and valid/ folders."
    DATA_YAML = found[0]

assert DATA_YAML and os.path.exists(DATA_YAML), f"data.yaml not found: {DATA_YAML}"
print("Using", DATA_YAML)

Roboflow and hand-made datasets often write relative paths like `../train/images` that break depending on where you unzip. This cell rewrites them as absolute paths, then counts boxes per class so you can spot classes with too few examples.

In [ ]:
import glob, collections, yaml

def fix_data_yaml(path):
    root = os.path.dirname(os.path.abspath(path))
    with open(path) as f:
        cfg = yaml.safe_load(f)
    base = cfg.get("path") or root
    if not os.path.isabs(base):
        base = os.path.normpath(os.path.join(root, base))
    for split in ("train", "val", "test"):
        value = cfg.get(split)
        if not value or not isinstance(value, str):
            continue
        trimmed = value.replace("../", "").replace("./", "")
        options = [value, os.path.join(base, value), os.path.join(root, value), os.path.join(root, trimmed), os.path.join(base, trimmed)]
        hit = next((os.path.normpath(o) for o in options if os.path.exists(o)), None)
        if hit:
            cfg[split] = hit
        elif split == "test":
            cfg.pop(split)
        else:
            raise FileNotFoundError(f"Couldn't find the {split} folder. Tried: {options}")
    if "val" not in cfg:
        raise ValueError("data.yaml needs a val (validation) split.")
    names = cfg["names"]
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names, key=int)]
    cfg["names"] = names
    cfg["nc"] = len(names)
    cfg.pop("path", None)
    fixed = os.path.join(root, "data.fixed.yaml")
    with open(fixed, "w") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    return fixed, cfg, [str(n) for n in names]

FIXED_YAML, DATA_CFG, NAMES = fix_data_yaml(DATA_YAML)

def label_dir(images_dir):
    head, tail = os.path.split(os.path.normpath(images_dir))
    return os.path.join(head, "labels") if tail == "images" else images_dir.replace("images", "labels")

for split in ("train", "val"):
    counts = collections.Counter()
    files_ = glob.glob(os.path.join(label_dir(DATA_CFG[split]), "*.txt"))
    for name in files_:
        with open(name) as f:
            for line in f:
                if line.strip():
                    counts[int(float(line.split()[0]))] += 1
    print(f"{split}: {len(files_)} label files")
    for i, n in enumerate(NAMES):
        print(f"   {i:>2} {n:<24} {counts.get(i, 0):>6} boxes")

## 3. Pick the threat classes

Boxes for these classes turn the bag red and put it on hold. Every other class is shown in blue as a normal object. The guess below matches common weapon and tool names; edit `THREATS` by hand if your dataset uses other names.

In [ ]:
THREAT_WORDS = [
    "gun", "pistol", "firearm", "rifle", "revolver", "knife", "blade", "dagger", "cutter", "razor",
    "scissor", "bullet", "ammo", "ammunition", "explosive", "grenade", "baton", "hammer", "wrench",
    "plier", "shuriken", "handcuff", "axe", "screwdriver",
]
THREATS = [n for n in NAMES if any(w in n.lower() for w in THREAT_WORDS)]
# THREATS = ["Gun", "Knife"]   # or set them by hand

print("Threat classes:", THREATS)
print("Other classes: ", [n for n in NAMES if n not in THREATS])

## 4. Train

`yolo11n` (nano) keeps in-browser inference well under a second on a laptop. `yolo11s` is more accurate but roughly 2–3× slower in the browser. X-ray bags can lie in any orientation, so vertical flips are turned on.

In [ ]:
from ultralytics import YOLO

BASE_MODEL = "yolo11n.pt"
EPOCHS = 50
IMGSZ = 640
BATCH = 16
RUNS = os.path.join(WORK, "runs")

model = YOLO(BASE_MODEL)
model.train(
    data=FIXED_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=15,
    flipud=0.5,
    cos_lr=True,
    plots=True,
    project=RUNS,
    name="xray",
    exist_ok=True,
)
BEST = str(model.trainer.best)
print("Best weights:", BEST)

## 5. Check accuracy

**Recall** on threat classes is the number that matters most for security: of the real knives and guns, how many did the model catch? **Precision** is how many alarms were real. mAP50 sums both up. Also try a few images from a *different* dataset or scanner; accuracy usually drops there, and judges will respect you saying so.

In [ ]:
best = YOLO(BEST)
metrics = best.val(data=FIXED_YAML, imgsz=IMGSZ, batch=BATCH, plots=True, project=RUNS, name="xray_val", exist_ok=True)
print(f"mAP50 {metrics.box.map50:.3f}   mAP50-95 {metrics.box.map:.3f}")
print(f"{'class':<24}{'precision':>10}{'recall':>8}{'AP50':>8}")
for i, c in enumerate(metrics.box.ap_class_index):
    p, r, ap50, _ = metrics.box.class_result(i)
    tag = "  threat" if NAMES[int(c)] in THREATS else ""
    print(f"{NAMES[int(c)]:<24}{p:>10.3f}{r:>8.3f}{ap50:>8.3f}{tag}")

try:
    from IPython.display import Image as IPImage, display
    display(IPImage(filename=os.path.join(str(metrics.save_dir), "confusion_matrix_normalized.png"), width=720))
except Exception as error:
    print("No confusion matrix image:", error)

## 6. Export for the browser

This writes `model.onnx` and `model.json` (class names, threat list, input size, output format), runs the exported file once to check its output shape, and zips both.

In [ ]:
import json, shutil, zipfile
import numpy as np
import onnxruntime as ort

DISPLAY_NAME = f"X-ray threats · {os.path.splitext(BASE_MODEL)[0].upper()}"

onnx_file = best.export(format="onnx", imgsz=IMGSZ, simplify=True, dynamic=False)
session = ort.InferenceSession(str(onnx_file))
inp, out = session.get_inputs()[0], session.get_outputs()[0]
output = session.run(None, {inp.name: np.zeros((1, 3, IMGSZ, IMGSZ), dtype=np.float32)})[0]
shape = list(output.shape)
if shape[1] == len(NAMES) + 4:
    fmt = "yolov8"
elif shape[-1] == 6:
    fmt = "end2end"
else:
    raise ValueError(f"Unexpected output shape {shape} for {len(NAMES)} classes")
print("input", inp.shape, "output", shape, "format", fmt)

config = {
    "name": DISPLAY_NAME,
    "file": "model.onnx",
    "inputSize": IMGSZ,
    "labels": NAMES,
    "threats": THREATS,
    "format": fmt,
    "defaultThreshold": 0.35,
    "iou": 0.45,
    "metrics": {"mAP50": round(float(metrics.box.map50), 3), "mAP50_95": round(float(metrics.box.map), 3)},
}
out_dir = os.path.join(WORK, "xray-model")
os.makedirs(out_dir, exist_ok=True)
shutil.copy(onnx_file, os.path.join(out_dir, "model.onnx"))
with open(os.path.join(out_dir, "model.json"), "w") as f:
    json.dump(config, f, indent=2)

ZIP = os.path.join(WORK, "xray-model.zip")
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for name in ("model.onnx", "model.json"):
        z.write(os.path.join(out_dir, name), name)
print("Wrote", ZIP, f"({os.path.getsize(ZIP) / 1e6:.1f} MB)")
print(json.dumps({k: v for k, v in config.items() if k != "labels"}, indent=2))

if IN_COLAB:
    from google.colab import files
    files.download(ZIP)

## 7. Put it in the console

1. Unzip `xray-model.zip` into `public/models/xray/` in the GILGAMESH repo, so you have `public/models/xray/model.onnx` and `public/models/xray/model.json`.
2. `npm run dev`, open the console, and switch to **Real X-rays**. The footer should name your model.
3. Drop in a few validation images (they're in the dataset's `valid/images` folder) and a few from somewhere else.
4. Tune **Alarm threshold** in the Assessment rail. Lower catches more threats and raises more false alarms. When you've picked a value, set `defaultThreshold` in `model.json` to match.